In [1]:
import numpy as np

# --- 1. SIMULATED MARKET DATA SETUP ---
np.random.seed(42)
num_assets = 4
num_days = 1000

# Generate random daily returns for 4 assets (Stocks, Bonds, Gold, Crypto)
mean_returns = np.array([0.12, 0.05, 0.07, 0.25]) / 252
volatilities = np.array([0.18, 0.05, 0.12, 0.50]) / np.sqrt(252)

simulated_returns = np.random.normal(mean_returns, volatilities, size=(num_days, num_assets))
risk_free_rate = 0.04 / 252

# --- 2. FITNESS FUNCTION (SHARPE RATIO) ---
def calculate_sharpe_ratio(weights):
    # Enforce constraints: sum up to 1.0
    weights = weights / np.sum(weights)

    port_returns = np.dot(simulated_returns, weights)
    exp_return = np.mean(port_returns)
    std_dev = np.std(port_returns)

    if std_dev == 0:
        return -9999

    sharpe = (exp_return - risk_free_rate) / std_dev
    return sharpe * np.sqrt(252)

# --- 3. GREY WOLF OPTIMIZER ALGORITHM ---
def grey_wolf_portfolio(n=25, max_iter=500):
    # Initialize the pack using Dirichlet distribution (guarantees positive values summing to 1)
    wolves = np.random.dirichlet(np.ones(num_assets), size=n)

    # Initialize leadership structures
    X_alpha = np.zeros(num_assets)
    X_beta = np.zeros(num_assets)
    X_delta = np.zeros(num_assets)

    alpha_score = -float('inf')
    beta_score = -float('inf')
    delta_score = -float('inf')

    for t in range(max_iter):
        # Update leadership rankings based on fitness evaluations
        for i in range(n):
            fitness = calculate_sharpe_ratio(wolves[i])

            if fitness > alpha_score:
                delta_score = beta_score
                X_delta = X_beta.copy()
                beta_score = alpha_score
                X_beta = X_alpha.copy()
                alpha_score = fitness
                X_alpha = wolves[i].copy()

            elif fitness > beta_score:
                delta_score = beta_score
                X_delta = X_beta.copy()
                beta_score = fitness
                X_beta = wolves[i].copy()

            elif fitness > delta_score:
                delta_score = fitness
                X_delta = wolves[i].copy()

        # Linearly decrease exploration factor 'a' from 2 to 0
        a = 2.0 - t * (2.0 / max_iter)

        # Position updating phase for all Omegas
        for i in range(n):
            # Trajectory adjustment relative to Alpha wolf
            r1, r2 = np.random.rand(num_assets), np.random.rand(num_assets)
            A1 = 2 * a * r1 - a
            C1 = 2 * r2
            D_alpha = np.abs(C1 * X_alpha - wolves[i])
            X1 = X_alpha - A1 * D_alpha

            # Trajectory adjustment relative to Beta wolf
            r1, r2 = np.random.rand(num_assets), np.random.rand(num_assets)
            A2 = 2 * a * r1 - a
            C2 = 2 * r2
            D_beta = np.abs(C2 * X_beta - wolves[i])
            X2 = X_beta - A2 * D_beta

            # Trajectory adjustment relative to Delta wolf
            r1, r2 = np.random.rand(num_assets), np.random.rand(num_assets)
            A3 = 2 * a * r1 - a
            C3 = 2 * r2
            D_delta = np.abs(C3 * X_delta - wolves[i])
            X3 = X_delta - A3 * D_delta

            # Average vector movements
            new_position = (X1 + X2 + X3) / 3.0

            # Repair violations (Ensure positive percentages and precise 100% allocation)
            new_position = np.abs(new_position)
            wolves[i] = new_position / np.sum(new_position)

    return X_alpha, alpha_score

# --- 4. EXECUTION & RESULTS ---
optimal_weights, best_sharpe = grey_wolf_portfolio()

print("=== OPTIMAL PORTFOLIO ALLOCATION (GWO) ===")
asset_names = ["Stocks", "Bonds", "Gold", "Crypto"]
for name, weight in zip(asset_names, optimal_weights):
    print(f"{name}: {weight * 100:.2f}%")

print(f"\nMaximized Annualized Sharpe Ratio: {best_sharpe:.4f}")


=== OPTIMAL PORTFOLIO ALLOCATION (GWO) ===
Stocks: 28.26%
Bonds: 56.88%
Gold: 5.92%
Crypto: 8.94%

Maximized Annualized Sharpe Ratio: 1.4562
